## Retrieve Under Utilized Lambda Functions

In [9]:
import boto3
from datetime import datetime, timedelta
import pytz
import os

from botocore.exceptions import NoCredentialsError

# Get credentials from environment variables
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_session_token = os.getenv('AWS_SESSION_TOKEN')  # Optional, for temporary credentials

region_name = os.getenv('AWS_DEFAULT_REGION', 'ap-southeast-1')  # Default to 'ap-southeast-1' if not set

# Validate credentials
if not aws_access_key_id or not aws_secret_access_key:
    raise NoCredentialsError

In [10]:
# Initialize clients
lambda_client = boto3.client(
    'lambda',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region_name
)


cloudwatch_client = boto3.client(
    'cloudwatch',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region_name
)

# set Singapore timezone
timezone = pytz.timezone('Asia/Singapore')

currrent_datetime = datetime.now(tz=timezone)

# Get the date 3 months ago
three_months_ago = currrent_datetime - timedelta(days=90)

In [11]:
def get_unused_lambdas():
    unused_functions = []
    functions = lambda_client.list_functions()['Functions']

    for function in functions:
        function_name = function['FunctionName']

        # Check invocation metrics
        response = cloudwatch_client.get_metric_data(
            MetricDataQueries=[
                {
                    'Id': 'invocations',
                    'MetricStat': {
                        'Metric': {
                            'Namespace': 'AWS/Lambda',
                            'MetricName': 'Invocations',
                            'Dimensions': [
                                {'Name': 'FunctionName', 'Value': function_name}
                            ]
                        },
                        'Period': 86400,
                        'Stat': 'Sum'
                    },
                    'ReturnData': True
                }
            ],
            StartTime=three_months_ago,
            EndTime=currrent_datetime
        )

        # Check if invocations exist
        data_points = response['MetricDataResults'][0]['Values']
        if not data_points or sum(data_points) == 0:
            unused_functions.append(function_name)

    return unused_functions

In [12]:
unused = get_unused_lambdas()
print("Unused Lambda Functions (Last 3 Months):")
for func in unused:
    print(func)

Unused Lambda Functions (Last 3 Months):
location-search-distributor-v2-my-pois-full-import
developer-listing-rec-biz-layer-recsys
data-recsys-biz-sls-my-api
titan-location-search-es-AWS679f53fac002430cb0da5b-bo1aZyahXp7G
test-sqs-output
data-feedback-classify-biz-dev-custom-resource-apigw-cw-role
ds-lambda-etl-ldp-user-profile-dev
serverless-api-template-ping
test-raas-stream-transformer
sam-test-deploy-2-dev-hello-world-yeah-function
regional-transaction-lambda-dev-transaction
test_download_bds_image
askguru-dev-dynamic-tagging
test-mobility-rahul
data-recsys-inf-multi-lang-encod299845be45c5c6d42d0febc84636bef0
retinafaces-detection-dev-inference-model
dummy-service-v1-dev-function
data-recsys-biz-sls-vn-api
data-recsys-inf-sls-id-dev-custom-resource-apigw-cw-role
test-secret-manager-sam-1
data-gcp-access-aws-secret-dev-custom-resource-apigw-cw-role
lambda-test-lby
data-img-blur-detect-api
test-tanvir-streaming-for-notebook
test-from-sqs-send
data-recsys-biz-sls-vn-dev-custom-resourc

In [13]:
for func in unused:
    print(f"- {func}")

- location-search-distributor-v2-my-pois-full-import
- developer-listing-rec-biz-layer-recsys
- data-recsys-biz-sls-my-api
- titan-location-search-es-AWS679f53fac002430cb0da5b-bo1aZyahXp7G
- test-sqs-output
- data-feedback-classify-biz-dev-custom-resource-apigw-cw-role
- ds-lambda-etl-ldp-user-profile-dev
- serverless-api-template-ping
- test-raas-stream-transformer
- sam-test-deploy-2-dev-hello-world-yeah-function
- regional-transaction-lambda-dev-transaction
- test_download_bds_image
- askguru-dev-dynamic-tagging
- test-mobility-rahul
- data-recsys-inf-multi-lang-encod299845be45c5c6d42d0febc84636bef0
- retinafaces-detection-dev-inference-model
- dummy-service-v1-dev-function
- data-recsys-biz-sls-vn-api
- data-recsys-inf-sls-id-dev-custom-resource-apigw-cw-role
- test-secret-manager-sam-1
- data-gcp-access-aws-secret-dev-custom-resource-apigw-cw-role
- lambda-test-lby
- data-img-blur-detect-api
- test-tanvir-streaming-for-notebook
- test-from-sqs-send
- data-recsys-biz-sls-vn-dev-cus

# Try Another retrieval

ChatGPT Prompt: from the lambda metrics, retrieve the metrics, i.e. last time invoked (in date granularity), total invokation in the last month, and sort them


In [33]:
# Get the current datetime
now = datetime.now(tz=timezone)
start_time = now - timedelta(days=90)  # Start time: 30 days ago

def get_lambda_metrics():
    functions = lambda_client.list_functions()['Functions']
    results = []

    for function in functions:
        function_name = function['FunctionName']

        # Get the total invocations in the last 30 days
        invocations = cloudwatch_client.get_metric_statistics(
            Namespace='AWS/Lambda',
            MetricName='Invocations',
            Dimensions=[
                {'Name': 'FunctionName', 'Value': function_name}
            ],
            StartTime=start_time,
            EndTime=now,
            Period=14*86400,  # 14 days: 2 weeks
            Statistics=['Sum']
        )

        # Calculate total invocations
        total_invocations = sum(dp['Sum'] for dp in invocations['Datapoints'])

        # Get the last invocation time
        if invocations['Datapoints']:
            last_invocation_time = max(dp['Timestamp'] for dp in invocations['Datapoints'])
        else:
            last_invocation_time = None

        results.append({
            'FunctionName': function_name,
            'LastInvocationTime': last_invocation_time,
            'TotalInvocations': total_invocations
        })

    # Sort by total invocations descending, then by last invocation time descending
    results.sort(
        key=lambda x: (
            -x['TotalInvocations'], x['LastInvocationTime'] if x['LastInvocationTime'] else datetime.min
        ),
        reverse=True
    )
    return results

In [34]:
import pandas as pd

metrics = get_lambda_metrics()

# Convert results to a DataFrame
df = pd.DataFrame(metrics)
df.head()

# Save DataFrame to CSV locally
csv_file_path = "lambda_metrics.csv"
df.to_csv(csv_file_path, index=False)


print(f"Metrics saved to {csv_file_path}.")


Metrics saved to lambda_metrics.csv.
